# MMT-BERT Training

Three-phase training of the MMT-BERT system (arXiv 2409.00919) over compound events:

1. **Phase (a)** — MLE-pretrain the **MMT generator** (decoder) with mixed standard + FIM objectives (`fim_ratio=0.5` keeps the product's infill capability).
2. **Phase (b)** — MLM-pretrain the **MusicBERT discriminator** (bidirectional encoder) with masked-event prediction.
3. **Phase (c)** — **GAN fine-tune** the generator against the discriminator with the relativistic standard loss + CE anchor. Results go to a *new* checkpoint (`MMT-GAN-…`) so the MLE checkpoint survives GAN instability.

> **Note:** the first cache build after switching to the compound-event schema reparses every score **and runs per-bar chord analysis** (music21 chordify) — expect it to be slow once; later runs are incremental.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("/notebooks/"))
sys.path.insert(0, os.path.abspath("../"))

import torch
from pathlib import Path
from ml.datasets import download_sources, load_paths, load_records, dataset_status
from ml.preprocess import build_token_cache, DEFAULT_CACHE_DIR
from ml.model import build_model, load_model, append_epoch_to_config
from ml.infill_trainer import train_infill
from ml.mlm_trainer import train_musicbert_mlm
from ml.gan_trainer import train_gan
from ml.events import FIELD_VOCAB_SIZES, FIELD_NAMES

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
print("Compound-event fields:", dict(zip(FIELD_NAMES, FIELD_VOCAB_SIZES)))

In [ ]:
# --- Checkpoint names --------------------------------------------------------
GEN_NAME  = "MMT-GEN-D512-L6"    # phase (a): MLE generator (standard + FIM)
DISC_NAME = "MMT-BERT-D512-L6"   # phase (b): MusicBERT discriminator (MLM)
GAN_NAME  = "MMT-GAN-D512-L6"    # phase (c): GAN-tuned generator (the API default)

# --- Per-phase schedules -----------------------------------------------------
GEN_EPOCHS,  GEN_BATCH,  GEN_LR  = 2000, 8, 3e-4
MLM_EPOCHS,  MLM_BATCH,  MLM_LR  = 500, 8, 1e-4
GAN_EPOCHS,  GAN_BATCH           = 100, 8
GAN_OPTIMIZER, GAN_LR_G, GAN_LR_D = "adagrad", 1e-2, 1e-3   # paper; or ("adamw", 1e-5, 1e-5)
LAMBDA_CE = 4.0                  # CE anchor weight in the generator GAN loss
FIM_RATIO = 0.5                  # share of FIM (infill) examples in (a) and (c)

# --- Dataset sources ---------------------------------------------------------
# Add "pdmx" / "wikifonia" to train on them (they download on first run).
SOURCES = ["openscore"]

# OpenScore: how many scores to pull from the GitHub repo (it has ~1462).
OPENSCORE_MAX = 10000

# PDMX (~250K, Zenodo 14648209): auto-downloads PDMX.csv + mxl.tar.gz (~2 GB) and
# extracts ONLY the selected subset's .mxl files.
PDMX_ROOT = None       # already-extracted dir, or None to download from Zenodo
PDMX_MAX = 20000

# Wikifonia lead sheets (melody + chord symbols): defaults to OpenEWLD.
WIKIFONIA_ROOT = None  # already-extracted dir, or None
WIKIFONIA_URL = None   # zip/tar URL, or None to use the OpenEWLD default

# Token-cache build: music21 parse processes (Linux fork -> safe to raise).
CACHE_WORKERS = 4

# Chord events for scores without chord symbols: "auto" analyzes one chord per
# bar via music21 chordify (slow, cache-build-time only); "existing" keeps only
# real symbols (e.g. Wikifonia); "off" strips all chords.
CHORDS_MODE = "auto"

In [ ]:
# Download / locate each selected source (per-source kwargs), then merge paths.
_noprog = lambda i, n: None
for src in SOURCES:
    if src == "openscore":
        download_sources(["openscore"], log=print, progress=_noprog, max_files=OPENSCORE_MAX)
    elif src == "pdmx":
        download_sources(["pdmx"], log=print, progress=_noprog,
                         max_files=PDMX_MAX, pdmx_root=PDMX_ROOT)
    elif src == "wikifonia":
        download_sources(["wikifonia"], log=print, progress=_noprog,
                         wikifonia_root=WIKIFONIA_ROOT, archive_url=WIKIFONIA_URL)

print(dataset_status(SOURCES))
paths = load_paths(SOURCES)
print(f"Using {len(paths)} score files across {SOURCES}.")

In [ ]:
# Parse every score ONCE into the on-disk token cache (incremental + parallel).
# The schema fingerprint changed with the MMT-BERT rewrite, so the first run
# rebuilds from scratch (including per-bar chord analysis when CHORDS_MODE=auto).
store = build_token_cache(load_records(SOURCES), n_workers=CACHE_WORKERS, chords=CHORDS_MODE)
print(f"Token cache ready: {len(store)} scores at {DEFAULT_CACHE_DIR}")

<hr>

## Phase (a) — Generator MLE pretraining

Decoder-only MMT generator, per-field cross-entropy, mixed standard + FIM examples (transpose augmentation s ~ U(−5, 6) per the paper).

In [ ]:
print(f"Building {GEN_NAME}...")
gen_model, gen_cfg = build_model(GEN_NAME, model_type="mmt_gen")
print(f"Generator parameters: {gen_model.num_params():,}")
print({k: gen_cfg[k] for k in ('d_model', 'n_heads', 'n_layers', 'max_seq_len')})

def make_on_epoch(name, cfg):
    def on_epoch(epoch, train_loss, val_loss):
        print(f"  -> epoch {epoch}  train={train_loss:.4f}  val={val_loss:.4f}")
        append_epoch_to_config(name, cfg, train_loss, val_loss)
    return on_epoch

In [ ]:
stop_flag = [False]  # set stop_flag[0] = True to interrupt training
train_infill(
    name=GEN_NAME,
    model=gen_model,
    data_paths=paths,
    epochs=GEN_EPOCHS,
    batch_size=GEN_BATCH,
    lr=GEN_LR,
    device=DEVICE,
    log=print,
    on_epoch=make_on_epoch(GEN_NAME, gen_cfg),
    stop_flag=stop_flag,
    fim_ratio=FIM_RATIO,
    resume_optimizer=False,
    cache_workers=CACHE_WORKERS,
)

<hr>

## Phase (b) — Discriminator MLM pretraining

MusicBERT-style bidirectional encoder pretrained with masked-event prediction (15% of events replaced by a learned [MASK]; `mask_scheme="bar"` masks whole bars, MusicBERT's anti-leakage variant).

In [ ]:
print(f"Building {DISC_NAME}...")
disc_model, disc_cfg = build_model(DISC_NAME, model_type="musicbert_disc")
print(f"Discriminator parameters: {disc_model.num_params():,}")

stop_flag_b = [False]
train_musicbert_mlm(
    name=DISC_NAME,
    model=disc_model,
    data_paths=paths,
    epochs=MLM_EPOCHS,
    batch_size=MLM_BATCH,
    lr=MLM_LR,
    device=DEVICE,
    log=print,
    on_epoch=make_on_epoch(DISC_NAME, disc_cfg),
    stop_flag=stop_flag_b,
    mask_prob=0.15,
    mask_scheme="event",   # or "bar"
    cache_workers=CACHE_WORKERS,
)

<hr>

## Phase (c) — GAN fine-tuning

Relativistic standard GAN (softplus form) + CE anchor; teacher-forced soft embeddings with straight-through Gumbel-softmax carry gradients through the discrete tokens. Writes to **new** checkpoints `MMT-GAN-…` / `MMT-GAN-…-DISC`.

Monitoring: the plotted curve for the GAN model is (train **G loss**, val **CE**) — if val CE climbs more than ~10% above the phase-(a) baseline, stop (`stop_flag_c[0] = True`) and keep an earlier checkpoint.

In [ ]:
import json
from ml.model import model_paths

stop_flag_c = [False]

def gan_on_epoch(epoch, g_loss, val_ce):
    print(f"  -> epoch {epoch}  G={g_loss:.4f}  val_CE={val_ce:.4f}")
    cfg_now = json.loads(model_paths(GAN_NAME)["config"].read_text())
    append_epoch_to_config(GAN_NAME, cfg_now, g_loss, val_ce)

train_gan(
    gen_name=GEN_NAME,
    disc_name=DISC_NAME,
    out_name=GAN_NAME,
    data_paths=paths,
    epochs=GAN_EPOCHS,
    batch_size=GAN_BATCH,
    device=DEVICE,
    log=print,
    on_epoch=gan_on_epoch,
    stop_flag=stop_flag_c,
    lambda_ce=LAMBDA_CE,
    optimizer=GAN_OPTIMIZER,
    lr_g=GAN_LR_G,
    lr_d=GAN_LR_D,
    gumbel=True,
    fim_ratio=FIM_RATIO,
    resume=False,          # True to continue a previous GAN run
    cache_workers=CACHE_WORKERS,
)
print(f"\nDone. The API serves '{GAN_NAME}' by default (api/schemas.py).")